In [ ]:
from claude_agent_sdk import ClaudeSDKClient
from pathlib import Path
import os
import pandas as pd
from src.agent_profiles import proposer_options, skill_generator_options, base_agent_options
from src.schemas import AgentResponse, ProposerResponse, ToolGeneratorResponse, ToolGeneratorResponse

dataset_path = Path("~/mnt/shared-resources-sentient-research/salah_resources/datasets/officeqa/")
csv_path = os.path.join(dataset_path, "officeqa.csv")


train_data = pd.read_csv(csv_path)
hard_data = train_data[train_data['difficulty'] == 'hard']
easy_data = train_data[train_data['difficulty'] == 'easy']


In [ ]:
sample_question = hard_data[hard_data.uid == "UID0123"] #.sample(10, random_state=55)
question = sample_question.question.values[0]
answer = sample_question.answer.values[0]
difficulty = sample_question.difficulty.values[0]

### Base Agent run

In [ ]:
async with ClaudeSDKClient(base_agent_options) as client:                                                                                             
    await client.query(question)                                                  
    response = [msg async for msg in client.receive_response()]
    output = AgentResponse.model_validate(response[-1].structured_output)
    reasoning = output.reasoning
    final_answer = output.final_answer

In [ ]:
print(reasoning)

In [ ]:
print(final_answer)
print(answer)

### Proposer Agent run

Looks at the agent trace, agent answer and ground truth; and proposes a new skill. 

In this case a few additions can be made:

1. Decide whether skill needs adding or prompt needs to be optimized
2. We can also add the reasoning/justification of the base agent to reach that answer
3. We should also decide whether an entirely new skill needs to be added or an existing skill needs to be modified.
4. We need to add memory/things that have been tried out for the program at hand.

In [ ]:
proposer_query = f"Agent trace: {str(output)}\n\nAgent Answer: {final_answer}\n\nGround Truth: {answer}"
async with ClaudeSDKClient(options=proposer_options) as client:                                                                                             
    await client.query(proposer_query)                                                                                                           
    proposer_response = [msg async for msg in client.receive_response()] 
    proposer_output = ProposerResponse.model_validate(proposer_response[-1].structured_output)
    proposer_tool_or_skill = proposer_output.proposed_tool_or_skill
    proposer_justification = proposer_output.justification

In [ ]:
print(proposer_tool_or_skill)
print(proposer_justification)
### Skill Generator Agent run


### Skill generator

1. Here, if a new skill is added that nullifies previous ones, then the agent should also delete the nullified skills so it doesn't  

In [ ]:
skill_generator_query = f"Proposed tool or skill: {proposer_tool_or_skill}\n\nJustification: {proposer_justification}"
async with ClaudeSDKClient(options=skill_generator_options) as client:                                                                                             
    await client.query(skill_generator_query)                                                                                                           
    skill_generator_response = [msg async for msg in client.receive_response()] 
    skill_generator_output = ToolGeneratorResponse.model_validate(skill_generator_response[-1].structured_output)
    skill_generator_generated_skill = skill_generator_output.generated_skill
    skill_generator_reasoning = skill_generator_output.reasoning

In [ ]:
print(skill_generator_generated_skill)

In [ ]:
print(skill_generator_reasoning)